In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("Imports successful ✓")

Imports successful ✓


In [2]:
# Load SHAP feature importance
shap_df = pd.read_csv('../outputs/shap_feature_importance.csv')
top_drivers = shap_df.head(5)['feature'].tolist()
print("Top churn drivers loaded:", top_drivers)

# Simulate at-risk customers
at_risk_customers = pd.DataFrame({
    'CustomerID': ['CUST_001', 'CUST_002', 'CUST_003'],
    'ChurnProbability': [0.91, 0.87, 0.82],
    'Contract': ['Month-to-month', 'Month-to-month', 'One year'],
    'Tenure': [2, 5, 8],
    'MonthlyCharges': [85.5, 72.3, 90.1],
    'OnlineSecurity': ['No', 'No', 'Yes']
})

print("\nAt-Risk Customers This Week:")
print(at_risk_customers)

Top churn drivers loaded: ['Contract', 'tenure', 'MonthlyCharges', 'TotalCharges', 'OnlineSecurity']

At-Risk Customers This Week:
  CustomerID  ChurnProbability        Contract  Tenure  MonthlyCharges  \
0   CUST_001              0.91  Month-to-month       2            85.5   
1   CUST_002              0.87  Month-to-month       5            72.3   
2   CUST_003              0.82        One year       8            90.1   

  OnlineSecurity  
0             No  
1             No  
2            Yes  


In [3]:
prompt_template = PromptTemplate(
    input_variables=["top_drivers", "at_risk_customers"],
    template="""
You are a senior retention analyst at a regional bank.

The churn prediction model has identified the following as the top drivers of customer churn:
{top_drivers}

The following customers are at high risk of churning this week:
{at_risk_customers}

Write a concise retention brief (max 150 words) for the retention manager that:
1. States how many customers are at risk and the estimated revenue at stake (assume $500 average monthly value per customer)
2. Explains the top 2 reasons these customers are likely to leave
3. Recommends one specific retention action per customer

Write in plain English. No jargon. Be direct and actionable.
"""
)

print("Prompt ready ✓")

Prompt ready ✓


In [4]:
llm = ChatOllama(model="llama3.2", temperature=0.3)
print("Ollama connected ✓")

Ollama connected ✓


In [5]:
chain = prompt_template | llm | StrOutputParser()

response = chain.invoke({
    "top_drivers": ", ".join(top_drivers),
    "at_risk_customers": at_risk_customers.to_string(index=False)
})

print("=== WEEKLY RETENTION BRIEF ===\n")
print(response)

=== WEEKLY RETENTION BRIEF ===

Retention Brief: High-Risk Customers

**Summary:**

We have identified 3 customers at high risk of churning this week, with estimated revenue at stake of $1,500 ($500 x 3 customers). Our churn prediction model has identified Contract and OnlineSecurity as the top drivers of customer churn for these customers.

**Customer Breakdown:**

* CUST_001: High risk due to contract and high monthly charges. Estimated revenue at stake: $500.
	+ Recommended action: Review and adjust contract terms to reduce monthly charges.
* CUST_002: High risk due to contract and high monthly charges. Estimated revenue at stake: $500.
	+ Recommended action: Review and adjust contract terms to reduce monthly charges.
* CUST_003: High risk due to low online security. Estimated revenue at stake: $500.
	+ Recommended action: Offer additional online security features or resources to improve security.

These actions are designed to address the top drivers of churn for each customer and 

In [6]:
with open('../outputs/retention_brief.txt', 'w') as f:
    f.write("=== WEEKLY RETENTION BRIEF ===\n\n")
    f.write(response)

print("Brief saved to outputs/retention_brief.txt ✓")

Brief saved to outputs/retention_brief.txt ✓


In [7]:
# Export data for Tableau
import sklearn
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# Load full dataset
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Export clean version for Tableau
tableau_df = df.copy()
tableau_df.to_csv('../outputs/churn_tableau_data.csv', index=False)

print("Tableau data exported ✓")
print(f"Rows: {tableau_df.shape[0]}, Columns: {tableau_df.shape[1]}")

Tableau data exported ✓
Rows: 7043, Columns: 21
